In [1]:
# ROS imports
import rospy
import actionlib
from geometry_msgs.msg import Pose, Twist
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
from assignment_2_2024.msg import PlanningAction, PlanningGoal, PositionVelocity
from actionlib_msgs.msg import GoalStatus

# Widgets and plotting
from ipywidgets import BoundedFloatText, Button, VBox, HBox, Box, Label, Output, Layout
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [2]:
# ---- Initialize ROS Node ----
rospy.init_node('notebook_client', anonymous=True)

# Action client
client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
client.wait_for_server()

True

In [3]:
# Global variables to store robot info
current_pos = {"x": 0.0, "y": 0.0}
robot_path = []
obstacle_distance = float('inf')
reached_goals = 0
cancelled_goals = 0

In [4]:
# ---- Widgets ----
x_input = BoundedFloatText(
    description='Target X:',
    value=0.0,
    min=-10.0,
    max=10.0,
    step=0.1
)

y_input = BoundedFloatText(
    description='Target Y:',
    value=0.0,
    min=-10.0,
    max=10.0,
    step=0.1
)
btn_set = Button(description='Set Goal', button_style='success')
btn_cancel = Button(description='Cancel Goal', button_style='danger')
info_pos = Label("Current Pos: (0.00, 0.00)")
info_vel = Label("Velocity: 0.00 lin / 0.00 ang")
info_obs = Label("Closest Obstacle: N/A")
goal_output = Output()

In [5]:
# ---- Callbacks ----
def odom_callback(msg):
    global current_pos, robot_path
    current_pos['x'] = msg.pose.pose.position.x
    current_pos['y'] = msg.pose.pose.position.y
    vx = msg.twist.twist.linear.x
    vz = msg.twist.twist.angular.z
    robot_path.append((current_pos['x'], current_pos['y']))

    info_pos.value = f"Current Pos: ({current_pos['x']:.2f}, {current_pos['y']:.2f})"
    info_vel.value = f"Velocity: {vx:.2f} lin / {vz:.2f} ang"

def scan_callback(msg):
    global obstacle_distance
    valid_ranges = [r for r in msg.ranges if r > 0.01]
    if valid_ranges:
        obstacle_distance = min(valid_ranges)
        info_obs.value = f"Closest Obstacle: {obstacle_distance:.2f} m"

In [6]:
# ---- Action Functions ----
def send_goal(btn):
    goal = PlanningGoal()
    goal.target_pose.pose.position.x = x_input.value
    goal.target_pose.pose.position.y = y_input.value
    client.send_goal(goal, done_cb=goal_done_callback)

    with goal_output:
        print(f"Sent goal to ({x_input.value}, {y_input.value})")

def goal_done_callback(state, result):
    global reached_goals

    # Callback when goal is done
    if state == GoalStatus.SUCCEEDED:
        reached_goals += 1
        with goal_output:
            print("Goal successfully reached")
    else:
        with goal_output:
            print(f"Goal finished with state: {state}")

def cancel_goal(btn):
    global cancelled_goals

    # Check if the goal is still active or pending
    state = client.get_state()
    if state in [GoalStatus.ACTIVE, GoalStatus.PENDING]:
        client.cancel_goal()
        cancelled_goals += 1
        with goal_output:
            print("Goal cancelled")
    else:
        with goal_output:
            print("No active goal to cancel")

In [7]:
# ---- Subscribers ----
rospy.Subscriber("/odom", Odometry, odom_callback)
rospy.Subscriber("/scan", LaserScan, scan_callback)

In [8]:
# ---- Plot robot's position and reached/cancelled goals ----
%matplotlib widget
fig, (ax_path, ax_bar) = plt.subplots(1, 2, figsize=(9, 4))

# Path plot
robot_line, = ax_path.plot([], [], 'b.-', label='Path')
ax_path.set_title("Robot Position")
ax_path.set_xlabel("X")
ax_path.set_ylabel("Y")
ax_path.set_xlim(10, -10)
ax_path.set_ylim(10, -10)
ax_path.grid(True)
ax_path.legend()

# Bar plot
bar_plot = ax_bar.bar(['Reached', 'Cancelled'], [0, 0], color=['green', 'red'])
ax_bar.set_ylim(0, 10)
ax_bar.set_title("Goal Status")

def update_plot(frame):
    if robot_path:
        xs, ys = zip(*robot_path)
        robot_line.set_data(xs, ys)
        ax_path.relim()
        ax_path.autoscale_view()

    bar_plot[0].set_height(reached_goals)
    bar_plot[1].set_height(cancelled_goals)
    ax_bar.set_ylim(0, max(10, reached_goals + cancelled_goals + 1))

ani = FuncAnimation(fig, update_plot, interval=500)

# ---- Widget Events ----
btn_set.on_click(send_goal)
btn_cancel.on_click(cancel_goal)

# ---- User Interface to set target, display info
spacer = Box(layout=Layout(width='30px'))
ui = VBox([
    HBox([
        VBox([x_input, y_input, HBox([btn_set, btn_cancel])]),
        spacer,
        VBox([info_pos,info_vel,info_obs])  
    ]),
    
    goal_output
])

display(ui)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …